# Phase 7: Advanced Sampling & Conclusion

*(This notebook is part of the GPT-2 Inference Engine from Scratch project.)*

## Lesson 36: Temperature, Top-K, and Top-P Sampling

**Goal:** Let's move away from greedy generation. We will write a custom sampler that supports Temperature (for randomness), Top-K, and Top-P to generate more natural and diverse text.

In [38]:
def sample_next_token(logits, temperature=1.0, top_k=0, top_p=1.0):
    logits = logits / temperature

    if top_k > 0:
        k = min(top_k, logits.shape[-1])
        kth_val = torch.topk(logits, k).values[-1]
        logits = torch.where(logits < kth_val, torch.full_like(logits, float("-inf")), logits)

    probs = torch.softmax(logits, dim=-1)

    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        cutoff = cumulative > top_p
        cutoff[1:] = cutoff[:-1].clone()
        cutoff[0] = False
        sorted_probs[cutoff] = 0.0
        probs = torch.zeros_like(probs).scatter(0, sorted_idx, sorted_probs)
        probs = probs / probs.sum()

    return torch.multinomial(probs, num_samples=1).item()

torch.manual_seed(42)
cache_sample = KVCache(len(blocks))
logits0 = prefill(ids, cache_sample)
next_id = sample_next_token(logits0[-1], temperature=0.8, top_k=40)

gen_ids_sampled = ids.tolist() + [next_id]
for _ in range(19):
    logits_step = decode_step(torch.tensor(next_id, device="cuda"), cache_sample)
    next_id = sample_next_token(logits_step[0], temperature=0.8, top_k=40)
    gen_ids_sampled.append(next_id)

print("Greedy (Lesson 20):         ", repr(tok.decode(gen_ids.tolist())))
print("Sampled (temp=0.8, top_k=40):", repr(tok.decode(gen_ids_sampled)))

Greedy (Lesson 20):          'The capital of France is the capital of the French Republic, and the capital'
Sampled (temp=0.8, top_k=40): 'The capital of France is home to several major industries, including aviation and energy. In some instances, it is not clear where'


# GPT-2 GPU Inference Engine (from scratch)
 
A hand-written GPU inference engine for GPT-2 , built to understand what production LLM
serving frameworks abstract away: memory movement, kernel-launch overhead, floating-point
precision, Triton kernel fusion, and quantization tradeoffs. No `model.generate()`, no
`torch.nn.MultiheadAttention` , every operation from token embedding to the final logit
was implemented by hand and verified against a HuggingFace reference model.
 
Built on Kaggle (Tesla T4, 16GB VRAM, no local GPU available). Inspired by
[Odin](https://github.com/) , a CPU-based `ggml`-backed inference engine studied first
as reference architecture, then rebuilt for GPU with the same design questions: how are
weights loaded, how does attention and the KV cache work, and what actually makes
inference fast or slow.
 
**Model:** GPT-2 (124M params, fp32) &nbsp;|&nbsp; **Hardware:** Tesla T4 (40 SMs, 2560 CUDA cores, 15.6 GB VRAM)
 
| Metric | Result |
| ,- | ,- |
| **TTFT** (5-token prompt) | 10.18 ms |
| **TPOT** (decode step) | 9.63 ms |
| **Decode throughput** | 103.8 tokens/sec |
| **Memory-bandwidth ceiling** | ~480 tokens/sec (22% reached , overhead-bound, not bandwidth-bound) |
| **KV cache growth** | ~0.074 MB/token (exact match to theory) |
| **Naive-recompute vs KV-cache crossover** | ~128 tokens of context |
| **Full-model correctness (fp32, 12 layers)** | 6.1e-5 max abs diff vs HuggingFace |
| **Fused LayerNorm (Triton)** | 2.47x isolated / 1.13x full-pipeline |
| **Fused attention (Triton)** | 2.35x isolated / 1.32x full-pipeline |
| **INT8 weight quantization** | 4.00x smaller, 0.39% max relative weight error |
| **Fused INT8 matvec kernel** | 2.28x faster than naive dequant, still 0.62x vs cuBLAS fp32 |
 
## Pipeline
 
```
model.safetensors ,load,> raw tensor dict (GPU, no HF model class)
                                    |
tokenizer.json ,tokenize,,,,->|
                                    v
                         prefill (all prompt tokens, fills KV cache)
                                    |
                         decode loop (1 token/step, reads+writes KV cache)
                                    |
                         sampler (temperature / top-k / top-p)
                                    |
                              generated text
```
 
## What was built
 
- **Weight loading** , raw `.safetensors` parsing, no HF model class; caught and fixed a
  substring-filter bug that silently dropped a real bias tensor (`"attn.bias" in key` also
  matched `attn.c_attn.bias`).
- **Manual forward pass** , embedding lookup, LayerNorm, packed-QKV attention (GPT-2's
  `Conv1D` layout, tied `lm_head`), causal masking, GELU-tanh MLP, residuals , all 12
  blocks, verified against HuggingFace logits at each stage.
- **KV cache** , prefill/decode split, exact (non-approximate) correctness, formal
  TTFT/TPOT measurement, and a context-length sweep (8 to 1023 tokens) proving the cache
  pays off once sequence length crosses the same ~128-token threshold where GPU matmuls
  stop being overhead-bound.
- **Triton kernels** , fused LayerNorm and a fused causal-attention kernel, both
  correctness-verified to fp32's precision floor (differences of exactly 1-1.5x the
  fp32 ULP, `2^-24`). Debugged two real Triton bugs: a `tl.dot` minimum-tile-size
  (`K >= 16`, a Tensor Core hardware constraint) and a silent stride-mismatch from
  `torch.empty_like()` copying a non-contiguous tensor's strides.
- **INT8 quantization** , symmetric per-tensor quantization with exact error-theory
  matches (max error = scale/2, MSE ~= scale^2/12), and a fused dequant+matvec Triton
  kernel matching the batch=1 decode pattern.
- **Sampling** , temperature, top-k, top-p on top of the greedy decoder.
## Engineering findings (the actual point of the project)
 
- **Kernel-launch overhead, not compute, dominates at small model scale.** A T4 has a
  fixed ~30 us floor per kernel launch. GPT-2's forward pass issues ~150-200 launches
  per token, which is why TTFT and TPOT came out nearly identical (10.18 ms vs 9.63 ms)
  despite processing very different amounts of work , and why decode throughput reached
  only 22% of the memory-bandwidth ceiling.
- **Floating-point error accumulates sub-linearly.** Stacking the same operation across
  12 layers grew the error from 1.14e-5 (one block) to 6.1e-5 (full model) , about 5x,
  not 12x , consistent with independent rounding errors partially cancelling
  (`sqrt(n)` growth, not `n`).
- **A "faster" kernel in isolation doesn't guarantee a faster pipeline.** Fused LayerNorm
  gave 2.47x alone but only 1.13x end-to-end, because LayerNorm was a minority share of
  total kernel launches , the majority were matmuls and attention that fusion hadn't
  touched yet.
- **Reading less data doesn't automatically mean going faster.** The INT8 fused kernel
  reads 4x fewer weight bytes but still lost to cuBLAS's fp32 matmul (0.62x), because it
  uses a plain reduction loop instead of Tensor Cores. Bandwidth savings only pay off
  once paired with compute-efficient kernel design , which is why production INT8
  kernels (Marlin, AWQ, GPTQ) are much more heavily engineered than this one.
- **Quantization error compounds through a matmul's reduction dimension.** A 0.39%
  weight-level error became a 6.24% output-level relative error after summing across
  768 terms, while cosine similarity stayed high (0.997) , most of the output vector's
  direction survived, but a few tail elements carried most of the error.
- **Benchmark noise is real, even with warm-up and best-of-N.** The same unchanged
  function measured 9.72 ms in one run and 11.17 ms a few cells later on the same Kaggle
  session , a reminder to trust interleaved, back-to-back comparisons over single
  point-in-time before/after numbers.
## Honest limitations
 
- Single sequence, batch size 1 only , no continuous batching or multi-request serving.
- GPT-2 124M only , architecture differs from Llama/Qwen (LayerNorm not RMSNorm,
  learned position embeddings not RoPE, plain GELU-MLP not gated SwiGLU).
- Quantization is naive per-tensor symmetric INT8 , no per-channel scales, no
  calibration (GPTQ/AWQ-style), and the kernel doesn't use Tensor Cores.
- All benchmark numbers are from a single, shared, free-tier Kaggle T4 , expect
  run-to-run variance of 10-15% on absolute timings.
## Possible next steps
 
- Per-channel INT8 quantization + a Tensor-Core-backed kernel to actually beat cuBLAS.
- Port the same pipeline to Qwen2.5-0.5B (RoPE, RMSNorm, gated SwiGLU) to compare against
  Odin's own CPU-side `forward.cpp` line by line.
- Naive batching experiment (batch size 1/2/4/8...) to revisit Lesson 7's weight-reuse
  finding inside the full generation loop, not just a single matmul.